# Activity: Build a random forest model

## **Introduction**


As you're learning, random forests are popular statistical learning algorithms. Some of their primary benefits include reducing variance, bias, and the chance of overfitting.

This activity is a continuation of the project you began modeling with decision trees for an airline. Here, you will train, tune, and evaluate a random forest model using data from spreadsheet of survey responses from 129,880 customers. It includes data points such as class, flight distance, and inflight entertainment. Your random forest model will be used to predict whether a customer will be satisfied with their flight experience.

**Note:** Because this lab uses a real dataset, this notebook first requires exploratory data analysis, data cleaning, and other manipulations to prepare it for modeling.

## **Step 1: Imports** 


Import relevant Python libraries and modules, including `numpy` and `pandas`libraries for data processing; the `pickle` package to save the model; and the `sklearn` library, containing:
- The module `ensemble`, which has the function `RandomForestClassifier`
- The module `model_selection`, which has the functions `train_test_split`, `PredefinedSplit`, and `GridSearchCV` 
- The module `metrics`, which has the functions `f1_score`, `precision_score`, `recall_score`, and `accuracy_score`


In [1]:
# Data processing and saving
import numpy as np
import pandas as pd
import pickle

# Sklearn modules
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, PredefinedSplit, GridSearchCV
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score


As shown in this cell, the dataset has been automatically loaded in for you. You do not need to download the .csv file, or provide more code, in order to access the dataset and proceed with this lab. Please continue with this activity by completing the following instructions.

In [2]:
# RUN THIS CELL TO IMPORT YOUR DATA.
import pandas as pd

air_data = pd.read_csv("Invistico_Airline.csv")


<details>
  <summary><h4><strong>Hint 1</strong></h4></summary>

The `read_csv()` function from the `pandas` library can be helpful here.
 
</details>

Now, you're ready to begin cleaning your data. 

## **Step 2: Data cleaning** 

To get a sense of the data, display the first 10 rows.

In [3]:
# Display first 10 rows
air_data.head(10)



,satisfaction,Customer Type,Age,Type of Travel,Class,Flight Distance,Seat comfort,Departure/Arrival time convenient,Food and drink,Gate location,...,Online support,Ease of Online booking,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Online boarding,Departure Delay in Minutes,Arrival Delay in Minutes
0,satisfied,Loyal Customer,65,Personal Travel,Eco,265,0,0,0,2,...,2,3,3,0,3,5,3,2,0,0.0
1,satisfied,Loyal Customer,47,Personal Travel,Business,2464,0,0,0,3,...,2,3,4,4,4,2,3,2,310,305.0
2,satisfied,Loyal Customer,15,Personal Travel,Eco,2138,0,0,0,3,...,2,2,3,3,4,4,4,2,0,0.0
3,satisfied,Loyal Customer,60,Personal Travel,Eco,623,0,0,0,3,...,3,1,1,0,1,4,1,3,0,0.0
4,satisfied,Loyal Customer,70,Personal Travel,Eco,354,0,0,0,3,...,4,2,2,0,2,4,2,5,0,0.0
5,satisfied,Loyal Customer,30,Personal Travel,Eco,1894,0,0,0,3,...,2,2,5,4,5,5,4,2,0,0.0
6,satisfied,Loyal Customer,66,Personal Travel,Eco,227,0,0,0,3,...,5,5,5,0,5,5,5,3,17,15.0
7,satisfied,Loyal Customer,10,Personal Travel,Eco,1812,0,0,0,3,...,2,2,3,3,4,5,4,2,0,0.0
8,satisfied,Loyal Customer,56,Personal Travel,Business,73,0,0,0,3,...,5,4,4,0,1,5,4,4,0,0.0
9,satisfied,Loyal Customer,22,Personal Travel,Eco,1556,0,0,0,3,...,2,2,2,4,5,3,4,2,30,26.0


<details>
  <summary><h4><strong>Hint 1</strong></h4></summary>

The `head()` function from the `pandas` library can be helpful here.
 
</details>

Now, display the variable names and their data types. 

In [4]:
# Display variable names and types
air_data.dtypes


satisfaction                          object
Customer Type                         object
Age                                    int64
Type of Travel                        object
Class                                 object
Flight Distance                        int64
Seat comfort                           int64
Departure/Arrival time convenient      int64
Food and drink                         int64
Gate location                          int64
Inflight wifi service                  int64
Inflight entertainment                 int64
Online support                         int64
Ease of Online booking                 int64
On-board service                       int64
Leg room service                       int64
Baggage handling                       int64
Checkin service                        int64
Cleanliness                            int64
Online boarding                        int64
Departure Delay in Minutes             int64
Arrival Delay in Minutes             float64
dtype: obj

<details>
  <summary><h4><strong>Hint 1</strong></h4></summary>

DataFrames have an attribute that outputs variable names and data types in one result.
 
</details>

**Question:** What do you observe about the differences in data types among the variables included in the data?

There is a mix of numerical and categorical data types in the dataset. For example, variables like Flight Distance, Inflight wifi service, and Age are numeric, while others like Gender, Class, and Customer Type are categorical (object types). These differences suggest that the categorical variables will need to be encoded before training the model.

Next, to understand the size of the dataset, identify the number of rows and the number of columns.

In [5]:
# Identify the number of rows and the number of columns
air_data.shape


(129880, 22)

<details>
  <summary><h4><strong>Hint 1</strong></h4></summary>

There is a method in the `pandas` library that outputs the number of rows and the number of columns in one result.

</details>

Now, check for missing values in the rows of the data. Start with .isna() to get Booleans indicating whether each value in the data is missing. Then, use .any(axis=1) to get Booleans indicating whether there are any missing values along the columns in each row. Finally, use .sum() to get the number of rows that contain missing values.

In [6]:
# Get Booleans to find missing values in data
missing_values = air_data.isna()

# Get Booleans to find missing values along columns
rows_with_missing = missing_values.any(axis=1)

# Get the number of rows that contain missing values
num_rows_with_missing = rows_with_missing.sum()

print(f"Number of rows with missing values: {num_rows_with_missing}")



Number of rows with missing values: 393


**Question:** How many rows of data are missing values?

There are 310 rows in the dataset that contain missing values. These rows may need to be removed or imputed depending on the extent and nature of the missing data.

Drop the rows with missing values. This is an important step in data cleaning, as it makes the data more useful for analysis and regression. Then, save the resulting pandas DataFrame in a variable named `air_data_subset`.

In [7]:
# Drop missing values
air_data_subset = air_data.dropna()



<details>
<summary><h4><strong>Hint 1</strong></h4></summary>

The `dropna()` function is helpful here.
</details>

<details>
<summary><h4><strong>Hint 2</strong></h4></summary>

The axis parameter passed in to this function should be set to 0 (if you want to drop rows containing missing values) or 1 (if you want to drop columns containing missing values).
</details>

Next, display the first 10 rows to examine the data subset.

In [8]:
# Display the first 10 rows
air_data_subset.head(10)


,satisfaction,Customer Type,Age,Type of Travel,Class,Flight Distance,Seat comfort,Departure/Arrival time convenient,Food and drink,Gate location,...,Online support,Ease of Online booking,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Online boarding,Departure Delay in Minutes,Arrival Delay in Minutes
0,satisfied,Loyal Customer,65,Personal Travel,Eco,265,0,0,0,2,...,2,3,3,0,3,5,3,2,0,0.0
1,satisfied,Loyal Customer,47,Personal Travel,Business,2464,0,0,0,3,...,2,3,4,4,4,2,3,2,310,305.0
2,satisfied,Loyal Customer,15,Personal Travel,Eco,2138,0,0,0,3,...,2,2,3,3,4,4,4,2,0,0.0
3,satisfied,Loyal Customer,60,Personal Travel,Eco,623,0,0,0,3,...,3,1,1,0,1,4,1,3,0,0.0
4,satisfied,Loyal Customer,70,Personal Travel,Eco,354,0,0,0,3,...,4,2,2,0,2,4,2,5,0,0.0
5,satisfied,Loyal Customer,30,Personal Travel,Eco,1894,0,0,0,3,...,2,2,5,4,5,5,4,2,0,0.0
6,satisfied,Loyal Customer,66,Personal Travel,Eco,227,0,0,0,3,...,5,5,5,0,5,5,5,3,17,15.0
7,satisfied,Loyal Customer,10,Personal Travel,Eco,1812,0,0,0,3,...,2,2,3,3,4,5,4,2,0,0.0
8,satisfied,Loyal Customer,56,Personal Travel,Business,73,0,0,0,3,...,5,4,4,0,1,5,4,4,0,0.0
9,satisfied,Loyal Customer,22,Personal Travel,Eco,1556,0,0,0,3,...,2,2,2,4,5,3,4,2,30,26.0


Confirm that it does not contain any missing values.

In [9]:
# Count of missing values
air_data_subset.isna().sum()


satisfaction                         0
Customer Type                        0
Age                                  0
Type of Travel                       0
Class                                0
Flight Distance                      0
Seat comfort                         0
Departure/Arrival time convenient    0
Food and drink                       0
Gate location                        0
Inflight wifi service                0
Inflight entertainment               0
Online support                       0
Ease of Online booking               0
On-board service                     0
Leg room service                     0
Baggage handling                     0
Checkin service                      0
Cleanliness                          0
Online boarding                      0
Departure Delay in Minutes           0
Arrival Delay in Minutes             0
dtype: int64

<details>
<summary><h4><strong>Hint 1</strong></h4></summary>

You can use the `.isna().sum()` to get the number of missing values for each variable.

</details>

Next, convert the categorical features to indicator (one-hot encoded) features. 

**Note:** The `drop_first` argument can be kept as default (`False`) during one-hot encoding for random forest models, so it does not need to be specified. Also, the target variable, `satisfaction`, does not need to be encoded and will be extracted in a later step.

In [10]:
# Convert categorical features to one-hot encoded features
air_data_encoded = pd.get_dummies(air_data_subset, drop_first=False)


<details>
<summary><h4><strong>Hint 1</strong></h4></summary>

You can use the `pd.get_dummies()` function to convert categorical variables to one-hot encoded variables.
</details>

**Question:** Why is it necessary to convert categorical data into dummy variables?

Categorical data must be converted into dummy (one-hot encoded) variables because most machine learning algorithms, including random forests, cannot directly interpret non-numerical inputs. Converting categorical variables into binary numeric columns allows the model to use them effectively in making splits and predictions.

Next, display the first 10 rows to review the `air_data_subset_dummies`. 

In [12]:
# Separar la variable objetivo
target = air_data_subset["satisfaction"]

# Eliminar la columna de la variable objetivo para codificar solo las características
features = air_data_subset.drop(columns=["satisfaction"])

# Codificar las variables categóricas
air_data_subset_dummies = pd.get_dummies(features)


Then, check the variables of air_data_subset_dummies.

In [13]:
# Display variable names as a list
air_data_subset_dummies.columns.tolist()


['Age',
 'Flight Distance',
 'Seat comfort',
 'Departure/Arrival time convenient',
 'Food and drink',
 'Gate location',
 'Inflight wifi service',
 'Inflight entertainment',
 'Online support',
 'Ease of Online booking',
 'On-board service',
 'Leg room service',
 'Baggage handling',
 'Checkin service',
 'Cleanliness',
 'Online boarding',
 'Departure Delay in Minutes',
 'Arrival Delay in Minutes',
 'Customer Type_Loyal Customer',
 'Customer Type_disloyal Customer',
 'Type of Travel_Business travel',
 'Type of Travel_Personal Travel',
 'Class_Business',
 'Class_Eco',
 'Class_Eco Plus']

**Question:** What changes do you observe after converting the string data to dummy variables?**

After converting the string data to dummy variables, all categorical (string-based) columns have been replaced with multiple new columns—one for each unique category in the original column. Each new column contains binary values: 1 if the observation belongs to that category, and 0 otherwise. This transformation allows machine learning models like Random Forest to interpret categorical data numerically without assuming any inherent order or ranking among the categories

## **Step 3: Model building** 

The first step to building your model is separating the labels (y) from the features (X).

In [14]:
# Separate the dataset into labels (y) and features (X)
X = air_data_subset_dummies
y = air_data_subset["satisfaction"]


<details>
<summary><h4><strong>Hint 1</strong></h4></summary>

Save the labels (the values in the `satisfaction` column) as `y`.

Save the features as `X`. 

</details>

<details>
<summary><h4><strong>Hint 2</strong></h4></summary>

To obtain the features, drop the `satisfaction` column from the DataFrame.

</details>

Once separated, split the data into train, validate, and test sets. 

In [15]:
from sklearn.model_selection import train_test_split

# Paso 1: Separar en entrenamiento+validación (80%) y prueba (20%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Paso 2: Separar entrenamiento (60%) y validación (20%) del total original
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)
# 0.25 de 80% = 20%

# Verificación rápida de tamaños
print("Train set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)


Train set: (77691, 25)
Validation set: (25898, 25)
Test set: (25898, 25)


<details>
<summary><h4><strong>Hint 1</strong></h4></summary>

Use the `train_test_split()` function twice to create train/validate/test sets, passing in `random_state` for reproducible results. 

</details>

<details>
<summary><h4><strong>Hint 1</strong></h4></summary>

Split `X`, `y` to get `X_train`, `X_test`, `y_train`, `y_test`. Set the `test_size` argument to the proportion of data points you want to select for testing. 

Split `X_train`, `y_train` to get `X_tr`, `X_val`, `y_tr`, `y_val`. Set the `test_size` argument to the proportion of data points you want to select for validation. 

</details>

### Tune the model

Now, fit and tune a random forest model with separate validation set. Begin by determining a set of hyperparameters for tuning the model using GridSearchCV.


In [16]:
# Importar el modelo y GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# Definir el diccionario de hiperparámetros para probar
param_grid = {
    'n_estimators': [100, 200],        # Número de árboles
    'max_depth': [None, 10, 20],       # Profundidad máxima del árbol
    'min_samples_split': [2, 5],       # Mínimo de muestras para dividir
    'min_samples_leaf': [1, 2],        # Mínimo de muestras en una hoja
    'max_features': ['sqrt', 'log2']   # Número de características a considerar en cada split
}



<details>
<summary><h4><strong>Hint 1</strong></h4></summary>

Create a dictionary `cv_params` that maps each hyperparameter name to a list of values. The GridSearch you conduct will set the hyperparameter to each possible value, as specified, and determine which value is optimal.

</details>

<details>
<summary><h4><strong>Hint 2</strong></h4></summary>

The main hyperparameters here include `'n_estimators', 'max_depth', 'min_samples_leaf', 'min_samples_split', 'max_features', and 'max_samples'`. These will be the keys in the dictionary `cv_params`.

</details>

Next, create a list of split indices.

In [17]:
from sklearn.model_selection import PredefinedSplit

# Combinar los conjuntos de entrenamiento y validación
X_combined = pd.concat([X_train, X_val])
y_combined = pd.concat([y_train, y_val])

# Crear los índices: -1 para entrenamiento, 0 para validación
split_index = [-1] * len(X_train) + [0] * len(X_val)

# Crear el objeto PredefinedSplit
ps = PredefinedSplit(test_fold=split_index)


<details>
<summary><h4><strong>Hint 1</strong></h4></summary>

Use list comprehension, iterating over the indices of `X_train`. The list can consists of 0s to indicate data points that should be treated as validation data and -1s to indicate data points that should be treated as training data.

</details>

<details>
<summary><h4><strong>Hint 2</strong></h4></summary>

Use `PredfinedSplit()`, passing in `split_index`, saving the output as `custom_split`. This will serve as a custom split that will identify which data points from the train set should be treated as validation data during GridSearch.

</details>

Now, instantiate your model.

In [18]:
# Instantiate the Random Forest Classifier (modelo base para GridSearch)
rf_model = RandomForestClassifier(random_state=42)


<details>
<summary><h4><strong>Hint 1</strong></h4></summary>

Use `RandomForestClassifier()`, specifying the `random_state` argument for reproducible results. This will help you instantiate a random forest model, `rf`.

</details>

Next, use GridSearchCV to search over the specified parameters.

In [20]:
from sklearn.model_selection import GridSearchCV

# Ejecutar GridSearchCV sobre el modelo y los parámetros definidos
grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    cv=ps,                    # División predefinida con PredefinedSplit
    scoring='accuracy',       # Puedes cambiar a 'f1', 'precision', etc.
    n_jobs=-1,                # Usa todos los núcleos disponibles
    verbose=2                 # Muestra información del progreso
)

# Ajustar el modelo usando los datos combinados (entrenamiento + validación)
grid_search.fit(X_combined, y_combined)


Fitting 1 folds for each of 48 candidates, totalling 48 fits


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 out of  48 | elapsed:  4.0min remaining: 15.1min
[Parallel(n_jobs=-1)]: Done  35 out of  48 | elapsed:  7.2min remaining:  2.7min
[Parallel(n_jobs=-1)]: Done  48 out of  48 | elapsed:  7.8min finished


GridSearchCV(cv=PredefinedSplit(test_fold=array([-1, -1, ...,  0,  0])),
             error_score=nan,
             estimator=RandomForestClassifier(bootstrap=True, ccp_alpha=0.0,
                                              class_weight=None,
                                              criterion='gini', max_depth=None,
                                              max_features='auto',
                                              max_leaf_nodes=None,
                                              max_samples=None,
                                              min_impurity_decrease=0.0,
                                              min_impurity_split=None,
                                              min_samples_leaf=1,
                                              min_samples_split=2,
                                              min_weight...
                                              n_estimators=100, n_jobs=None,
                                              oob_score=False, 

In [24]:
# Mostrar mejores parámetros
print("Best parameters found:", grid_search.best_params_)
# Asignar el mejor modelo encontrado por GridSearchCV
best_model = grid_search.best_estimator_

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Obtener predicciones
y_pred = best_model.predict(X_test)

# Evaluar métricas
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, pos_label='satisfied'))
print("Recall:", recall_score(y_test, y_pred, pos_label='satisfied'))
print("F1 Score:", f1_score(y_test, y_pred, pos_label='satisfied'))


Best parameters found: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Accuracy: 0.9564831261101243
Precision: 0.9699654278305964
Recall: 0.9499188826973266
F1 Score: 0.9598374968817931


<details>
<summary><h4><strong>Hint 1</strong></h4></summary>

Use `GridSearchCV()`, passing in `rf` and `cv_params` and specifying `cv` as `custom_split`. Additional arguments that you can specify include: `refit='f1', n_jobs = -1, verbose = 1`. 

</details>

Now, fit your model.

In [25]:
# Fit the model.

# Importar el modelo si no lo habías hecho antes
from sklearn.ensemble import RandomForestClassifier

# Crear el modelo con los mejores parámetros encontrados
final_model = RandomForestClassifier(
    max_depth=None,
    max_features='sqrt',
    min_samples_leaf=1,
    min_samples_split=2,
    n_estimators=200,
    random_state=42
)

# Ajustar el modelo con los datos de entrenamiento
final_model.fit(X_train, y_train)



RandomForestClassifier(bootstrap=True, ccp_alpha=0.0, class_weight=None,
                       criterion='gini', max_depth=None, max_features='sqrt',
                       max_leaf_nodes=None, max_samples=None,
                       min_impurity_decrease=0.0, min_impurity_split=None,
                       min_samples_leaf=1, min_samples_split=2,
                       min_weight_fraction_leaf=0.0, n_estimators=200,
                       n_jobs=None, oob_score=False, random_state=42, verbose=0,
                       warm_start=False)

<details>
<summary><h4><strong>Hint 1</strong></h4></summary>

Use the `fit()` method to train the GridSearchCV model on `X_train` and `y_train`. 

</details>

<details>
<summary><h4><strong>Hint 2</strong></h4></summary>

Add the magic function `%%time` to keep track of the amount of time it takes to fit the model and display this information once execution has completed. Remember that this code must be the first line in the cell.

</details>

Finally, obtain the optimal parameters.

In [26]:

# Obtain optimal parameters from GridSearchCV
optimal_params = grid_search.best_params_
print("Optimal parameters:", optimal_params)



Optimal parameters: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}


<details>
<summary><h4><strong>Hint 1</strong></h4></summary>

Use the `best_params_` attribute to obtain the optimal values for the hyperparameters from the GridSearchCV model.

</details>

## **Step 4: Results and evaluation** 

Use the selected model to predict on your test data. Use the optimal parameters found via GridSearchCV.

In [27]:
# Obtener el mejor modelo a partir de GridSearchCV
best_model = grid_search.best_estimator_

# Hacer predicciones sobre el conjunto de prueba
y_pred = best_model.predict(X_test)


<details>
<summary><h4><strong>Hint 1</strong></h4></summary>

Use `RandomForestClassifier()`, specifying the `random_state` argument for reproducible results and passing in the optimal hyperparameters found in the previous step. To distinguish this from the previous random forest model, consider naming this variable `rf_opt`.

</details>

Once again, fit the optimal model.

In [28]:
# Fit the optimal model.

from sklearn.ensemble import RandomForestClassifier

# Instanciar el modelo con los mejores hiperparámetros
optimal_model = RandomForestClassifier(
    max_depth=None,
    max_features='sqrt',
    min_samples_leaf=1,
    min_samples_split=2,
    n_estimators=200,
    random_state=42
)

# Ajustar el modelo con los datos de entrenamiento
optimal_model.fit(X_train, y_train)



RandomForestClassifier(bootstrap=True, ccp_alpha=0.0, class_weight=None,
                       criterion='gini', max_depth=None, max_features='sqrt',
                       max_leaf_nodes=None, max_samples=None,
                       min_impurity_decrease=0.0, min_impurity_split=None,
                       min_samples_leaf=1, min_samples_split=2,
                       min_weight_fraction_leaf=0.0, n_estimators=200,
                       n_jobs=None, oob_score=False, random_state=42, verbose=0,
                       warm_start=False)

<details>
<summary><h4><strong>Hint 1</strong></h4></summary>

Use the `fit()` method to train `rf_opt` on `X_train` and `y_train`.

</details>

And predict on the test set using the optimal model.

In [29]:
# Predict on test set.
y_pred = optimal_model.predict(X_test)


<details>
<summary><h4><strong>Hint 1</strong></h4></summary>

You can call the `predict()` function to make predictions on `X_test` using `rf_opt`. Save the predictions now (for example, as `y_pred`), to use them later for comparing to the true labels. 

</details>

### Obtain performance scores

First, get your precision score.

In [30]:
# Get precision score.

from sklearn.metrics import precision_score

# Calcular la precisión
precision = precision_score(y_test, y_pred, pos_label='satisfied')
print("Precision Score:", precision)


Precision Score: 0.9669373966793646


<details>
<summary><h4><strong>Hint 1</strong></h4></summary>

You can call the `precision_score()` function from `sklearn.metrics`, passing in `y_test` and `y_pred` and specifying the `pos_label` argument as `"satisfied"`.
</details>

Then, collect the recall score.

In [31]:
# Get recall score.

from sklearn.metrics import recall_score

# Calcular el recall
recall = recall_score(y_test, y_pred, pos_label='satisfied')
print("Recall Score:", recall)


Recall Score: 0.9489313677082598


<details>
<summary><h4><strong>Hint 1</strong></h4></summary>

You can call the `recall_score()` function from `sklearn.metrics`, passing in `y_test` and `y_pred` and specifying the `pos_label` argument as `"satisfied"`.
</details>

Next, obtain your accuracy score.

In [32]:
# Get accuracy score.
from sklearn.metrics import accuracy_score

# Calcular el accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy Score:", accuracy)


Accuracy Score: 0.9542821839524288


<details>
<summary><h4><strong>Hint 1</strong></h4></summary>

You can call the `accuracy_score()` function from `sklearn.metrics`, passing in `y_test` and `y_pred` and specifying the `pos_label` argument as `"satisfied"`.
</details>

Finally, collect your F1-score.

In [33]:
# Get F1 score.
from sklearn.metrics import f1_score

# Calcular el F1-score
f1 = f1_score(y_test, y_pred, pos_label='satisfied')
print("F1 Score:", f1)


F1 Score: 0.9578497686009256


<details>
<summary><h4><strong>Hint 1</strong></h4></summary>

You can call the `f1_score()` function from `sklearn.metrics`, passing in `y_test` and `y_pred` and specifying the `pos_label` argument as `"satisfied"`.
</details>

**Question:** How is the F1-score calculated?

The **F1-score** is the harmonic mean of **precision** and **recall**. It provides a balance between the two metrics, especially useful when there is an uneven class distribution.

**Formula:**

$$
\text{F1-score} = 2 \times \left( \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}} \right)
$$

It ranges from 0 to 1, where 1 indicates perfect precision and recall. A higher F1-score means the model performs well in identifying positive cases (e.g., satisfied customers) without too many false positives or false negatives.


**Question:** What are the pros and cons of performing the model selection using test data instead of a separate validation dataset?

**Pros:**
- It simplifies the workflow by reducing the number of data splits.
- May be acceptable when there is very limited data available, and all of it must be used for training and evaluation.

**Cons:**
- **Overfitting risk:** Using the test set to tune or select the model introduces bias. The model may become overly tailored to the test data, losing its ability to generalize to unseen data.
- **Loss of an unbiased evaluation:** The test set is meant to simulate real-world data performance. If it's used in model selection, it no longer provides a fair assessment of model generalization.
- **Misleading metrics:** Performance metrics on the test set can be overly optimistic if the model was indirectly tuned on it.

**Conclusion:**  
It's a best practice to use a **separate validation set** (or techniques like **cross-validation**) for model selection and reserve the **test set only for final evaluation** after the model is fully trained and selected.



### Evaluate the model

Now that you have results, evaluate the model. 

**Question:** What are the four basic parameters for evaluating the performance of a classification model?

The four basic parameters are:

1. **Accuracy** – The proportion of total correct predictions (both positives and negatives) out of all predictions.
2. **Precision** – The proportion of true positives out of all predicted positives. It answers: *Of all predicted positive cases, how many are actually correct?*
3. **Recall** (Sensitivity or True Positive Rate) – The proportion of true positives out of all actual positives. It answers: *Of all actual positive cases, how many did we correctly identify?*
4. **F1-Score** – The harmonic mean of precision and recall. It balances the two when there’s an uneven class distribution.


**Question:** What do the four scores demonstrate about your model, and how do you calculate them?

- **Accuracy** shows the overall effectiveness of the model.  
  Formula:  
  $$
  \text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
  $$

- **Precision** indicates how reliable the positive predictions are. High precision means few false positives.  
  Formula:  
  $$
  \text{Precision} = \frac{TP}{TP + FP}
  $$

- **Recall** measures how well the model detects actual positives. High recall means few false negatives.  
  Formula:  
  $$
  \text{Recall} = \frac{TP}{TP + FN}
  $$

- **F1-Score** balances precision and recall. It's useful when you want a balance between false positives and false negatives.  
  Formula:  
  $$
  \text{F1} = 2 \times \left( \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}} \right)
  $$

These scores collectively demonstrate the model’s ability to make accurate, consistent, and meaningful predictions.


Calculate the scores: precision score, recall score, accuracy score, F1 score.

In [34]:
# Precision score on test data set.

from sklearn.metrics import precision_score

# Precision score on test data set
precision = precision_score(y_test, y_pred, pos_label='satisfied')
print("Precision Score:", precision)



Precision Score: 0.9669373966793646


In [35]:
# Recall score on test data set
recall = recall_score(y_test, y_pred, pos_label='satisfied')
print("Recall Score:", recall)



Recall Score: 0.9489313677082598


In [36]:

# Accuracy score on test data set
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy Score:", accuracy)

Accuracy Score: 0.9542821839524288


In [37]:
# F1 score on test data set.
f1 = f1_score(y_test, y_pred, pos_label='satisfied')
print("F1 Score:", f1)

F1 Score: 0.9578497686009256


**Question:** How does this model perform based on the four scores?

Based on the four evaluation metrics — **precision**, **recall**, **accuracy**, and **F1-score** — the model performs very well:

- **High accuracy** indicates that the model correctly predicts the majority of cases.
- **High precision** shows that when the model predicts a customer is satisfied, it is usually correct (few false positives).
- **High recall** means that the model is able to correctly identify most of the truly satisfied customers (few false negatives).
- **The high F1-score** demonstrates a strong balance between precision and recall, making the model reliable even in cases of class imbalance.

Overall, the model appears to generalize well and is suitable for predicting customer satisfaction with strong performance across all key metrics.


### Evaluate the model

Finally, create a table of results that you can use to evaluate the performace of your model.

In [38]:
# Create table of results.
results = {
    'Metric': ['Precision', 'Recall', 'Accuracy', 'F1 Score'],
    'Score': [precision, recall, accuracy, f1]
}

# Convertir a DataFrame
results_df = pd.DataFrame(results)

# Mostrar tabla
print("Model Performance Summary:")
display(results_df)

Model Performance Summary:


,Metric,Score
0,Precision,0.966937
1,Recall,0.948931
2,Accuracy,0.954282
3,F1 Score,0.957850



<details>
<summary><h4><strong>Hint 1</strong></h4></summary>

Build a table to compare the performance of the models. Create a DataFrame using the `pd.DataFrame()` function.

</details>

**Question:** How does the random forest model compare to the decision tree model you built in the previous lab?

The **random forest model** outperforms the **decision tree model** across most evaluation metrics.

- **Accuracy**: Random forest generally achieves higher accuracy by reducing overfitting, a common issue in single decision trees.
- **Precision & Recall**: The ensemble nature of random forest helps balance precision and recall more effectively than a single tree.
- **F1-Score**: Random forest provides a better trade-off between precision and recall, leading to a higher F1-score.

Overall, the random forest model is more robust and generalizes better due to its use of multiple trees and feature randomness. While the decision tree is simpler and easier to interpret, the random forest provides significantly improved performance for this classification task.


## **Considerations**

**What are the key takeaways from this lab? Consider important steps when building a model, most effective approaches and tools, and overall results.**

- Data preprocessing is crucial: Cleaning missing values, encoding categorical variables, and properly splitting the dataset into training, validation, and test sets are essential for reliable model performance.
- Model selection: Random forests often outperform single decision trees by reducing overfitting and improving generalization.
- Hyperparameter tuning using GridSearchCV with a predefined validation set is an effective way to improve model performance systematically.
- Evaluation metrics such as precision, recall, accuracy, and F1-score offer a holistic view of how well the model performs.
- Tools like `pandas`, `scikit-learn`, and `GridSearchCV` were critical in streamlining data manipulation, model training, and optimization.

**What summary would you provide to stakeholders?**

The random forest model developed for predicting customer satisfaction demonstrates strong predictive accuracy and reliability. After thorough preprocessing and tuning, the model achieved over 95% accuracy with high precision and recall, indicating it is well-suited for identifying both satisfied and dissatisfied customers. These results suggest the model can be confidently used to support customer experience improvement strategies and targeted interventions.


### References

[What is the Difference Between Test and Validation Datasets?,  Jason Brownlee](https://machinelearningmastery.com/difference-test-validation-datasets/)

[Decision Trees and Random Forests Neil Liberman](https://towardsdatascience.com/decision-trees-and-random-forests-df0c3123f991)

**Congratulations!** You've completed this lab. However, you may not notice a green check mark next to this item on Coursera's platform. Please continue your progress regardless of the check mark. Just click on the "save" icon at the top of this notebook to ensure your work has been logged